## 📦 Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
import time
from itertools import combinations
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import faiss
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('../src')
from ingest import normalize_to_ingredient_rxcui

## 📂 Load Knowledge Base

In [2]:
kb_df = pd.read_csv('../data/processed_interactions_kb.csv')
kb_df['pair_key'] = kb_df['pair_key'].apply(eval)

print(f"✅ Loaded knowledge base:")
print(f"   Total interactions: {len(kb_df):,}")
print(f"   Unique drug pairs: {kb_df['pair_key'].nunique():,}")
print(f"\nSample:")
print(kb_df[['Drug 1', 'Drug 2', 'Interaction Description', 'pair_key']].head())

✅ Loaded knowledge base:
   Total interactions: 170,782
   Unique drug pairs: 170,782

Sample:
                Drug 1       Drug 2  \
0           Trioxsalen  Verteporfin   
1  Aminolevulinic acid  Verteporfin   
2     Titanium dioxide  Verteporfin   
3     Tiaprofenic acid  Verteporfin   
4          Cyamemazine  Verteporfin   

                             Interaction Description          pair_key  
0  Trioxsalen may increase the photosensitizing a...   (10844, 118886)  
1  Aminolevulinic acid may increase the photosens...     (118886, 683)  
2  Titanium dioxide may increase the photosensiti...   (118886, 38323)  
3  Tiaprofenic acid may increase the photosensiti...  (118886, 618442)  
4  Cyamemazine may increase the photosensitizing ...   (118886, 21877)  


## 📂 Load RxNorm Mappings

In [3]:
with open('../data/rxnorm_lookups.pkl', 'rb') as f:
    lookups = pickle.load(f)

name_to_rxcui = lookups['name_to_rxcui']
rxcui_to_names = lookups['rxcui_to_names']
bn_to_in_map = lookups['bn_to_in_map']
ingredient_name_to_rxcui = lookups['ingredient_name_to_rxcui']

print(f"✅ Loaded RxNorm mappings:")
print(f"   Name→RxCUI: {len(name_to_rxcui):,}")
print(f"   RxCUI→Names: {len(rxcui_to_names):,}")
print(f"   Brand→Ingredient: {len(bn_to_in_map):,}")
print(f"   Ingredient cache: {len(ingredient_name_to_rxcui):,}")

✅ Loaded RxNorm mappings:
   Name→RxCUI: 157,972
   RxCUI→Names: 82,134
   Brand→Ingredient: 77,518
   Ingredient cache: 6,409


## 🔧 Drug Normalization Function

In [4]:
def normalize_drug(drug_name):
    """Normalize drug name to ingredient RxCUI using ingestion logic"""
    rxcui, dtype = normalize_to_ingredient_rxcui(
        drug_name,
        name_to_rxcui,
        bn_to_in_map,
        ingredient_name_to_rxcui
    )
    return rxcui


print("🔍 Normalization tests:")
test_drugs = ['Tylenol', 'Acetaminophen', 'Advil', 'Ibuprofen', 'Warfarin']
for drug in test_drugs:
    rxcui = normalize_drug(drug)
    print(f"   {drug:15} → RxCUI {rxcui}")

🔍 Normalization tests:
   Tylenol         → RxCUI 161
   Acetaminophen   → RxCUI 161
   Advil           → RxCUI 5640
   Ibuprofen       → RxCUI 5640
   Warfarin        → RxCUI 11289


## 🧠 Build or Load FAISS Index

In [5]:
load_dotenv("../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

index_path = Path("../data/faiss_index.bin")
embeddings_path = Path("../data/interaction_embeddings.npy")


def get_openai_embeddings_batch(texts, model="text-embedding-3-large"):
    """Generate embeddings with retry logic and latency tracking"""
    for attempt in range(3):
        try:
            t0 = time.time()
            resp = client.embeddings.create(model=model, input=texts)
            latency = (time.time() - t0) * 1000
            print(f"Batch of {len(texts)} | Latency: {latency:.1f} ms")
            embeddings = [d.embedding for d in resp.data]
            return np.array(embeddings, dtype="float32")
        except Exception as e:
            print(f"⚠️ Retry {attempt + 1}/3 for batch after error: {e}")
            time.sleep(2)
    raise RuntimeError("Failed to embed batch after 3 retries.")


if index_path.exists() and embeddings_path.exists():
    print("📂 Loading existing FAISS index...")
    index = faiss.read_index(str(index_path))
    embeddings = np.load(str(embeddings_path))
    print(f"✅ Loaded index with {index.ntotal:,} vectors ({embeddings.shape[1]} dims)")
else:
    print("🔨 Building FAISS index (first run)...")
    texts = [
        f"{row['Drug 1']} and {row['Drug 2']} interaction: {row['Interaction Description']}"
        for _, row in kb_df.iterrows()
    ]
    print(f"Generating embeddings for {len(texts):,} interactions...")

    batch_size = 1000
    all_batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    start = time.time()

    all_embeddings = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(get_openai_embeddings_batch, batch): idx
                   for idx, batch in enumerate(all_batches, 1)}
        for future in as_completed(futures):
            batch_idx = futures[future]
            try:
                batch_embeddings = future.result()
                all_embeddings.append(batch_embeddings)
                print(f"✅ Completed batch {batch_idx}/{len(all_batches)}")

                if len(all_embeddings) % 5 == 0:
                    temp_embeddings = np.vstack(all_embeddings)
                    np.save(embeddings_path.with_suffix(".tmp.npy"), temp_embeddings)
                    print(f"💾 Checkpoint saved ({len(temp_embeddings):,} embeddings)")
            except Exception as e:
                print(f"❌ Batch {batch_idx} failed: {e}")

    embeddings = np.vstack(all_embeddings)
    print(f"✅ Generated {embeddings.shape[0]:,} embeddings ({embeddings.shape[1]} dims)")

    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    index_path.parent.mkdir(exist_ok=True)
    faiss.write_index(index, str(index_path))
    np.save(str(embeddings_path), embeddings)
    total_min = (time.time() - start) / 60
    print(f"✅ Saved FAISS index ({index.ntotal:,} vectors) in {total_min:.1f} min")

📂 Loading existing FAISS index...
✅ Loaded index with 170,782 vectors (3072 dims)
